# MPA Linker 转运率预测 — 随机森林 (v1.0)

本 Notebook 在 **Jupyter** 环境中复现随机森林建模全流程：清洗 → 训练 → 评估 → 特征重要性 → 可视化。

- **版本**：v1.0（git tag `v1.0.0`）
- **目标列**：`transport_rate`（转运率）
- **运行环境**：Python ≥ 3.9，依赖见仓库根目录 `requirements.txt`（已锁定版本）

**启动方式**（请在仓库根目录启动 Jupyter）：
```bash
jupyter notebook notebooks/MPA_RF_Training.ipynb
# 或
jupyter lab notebooks/MPA_RF_Training.ipynb
```

> 所有结果写入 `results/`，与 CLI 方式（`python run.py`）完全一致。切换到 v1.0 只需 `git checkout v1.0.0`。

In [ ]:
%matplotlib inline
import sys, subprocess, json, pathlib
from IPython.display import Image, display
print('Python', sys.version.split()[0])

## 1. 环境与数据

若 `data/processed/mpa_linkers_clean.csv` 不存在，而 `data/raw/` 下有原始 CSV，则先自动清洗。

In [ ]:
ROOT = pathlib.Path.cwd()
clean = ROOT / 'data' / 'processed' / 'mpa_linkers_clean.csv'
raw = next((p for p in (ROOT / 'data' / 'raw').glob('*.csv')), None)
if not clean.exists() and raw is not None:
    subprocess.run([sys.executable, '-m', 'src.clean_csv',
                    '--input', str(raw), '--output', str(clean)], check=True)
    print('cleaned raw ->', clean)
elif clean.exists():
    print('using', clean)
else:
    raise FileNotFoundError('未找到数据，请先放入 data/processed/ 或 data/raw/')

## 2. 训练随机森林

复用 `src/train.py`：可复现切分 + 5 折交叉验证 + 测试集评估 + 保存模型与图表。

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'src.train',
     '--data', str(clean),
     '--target', 'transport_rate',
     '--id-col', 'compound_name'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 3. 评估指标

In [ ]:
metrics = json.loads((ROOT / 'results' / 'metrics.json').read_text(encoding='utf-8'))
print('CV  R2 :', round(metrics['cross_validation']['r2_mean'], 3),
      '+/-', round(metrics['cross_validation']['r2_std'], 3))
print('Test R2 :', round(metrics['test_set']['r2'], 3))
print('Test MAE:', round(metrics['test_set']['mae'], 3))
print('Test RMSE:', round(metrics['test_set']['rmse'], 3))

## 4. 可视化

In [ ]:
for img in ['feature_importance.png', 'predicted_vs_actual.png']:
    p = ROOT / 'results' / img
    if p.exists():
        display(Image(filename=str(p)))

## 5. 全量复核

In [ ]:
r = subprocess.run([sys.executable, '-m', 'src.evaluate',
                    '--target', 'transport_rate', '--id-col', 'compound_name'],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)